# Association Rule Mining C1 — Streaming Series

## Role 1: Setup, EDA and Transaction Building

This section prepares the streaming series dataset for Association Rule Mining. 
It covers initial exploratory data analysis, construction of viewing-session 
transactions, basket-size analysis, and creation of a separate show lookup table.

In [3]:
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)

In [4]:
df = pd.read_csv("../data/streaming_series_watched.csv")

df.head()

,ls dataviewing_session_id,transaction_date,show_id,item_name,category,avg_episode_minutes
0,1,2025-05-07,S05,Office Antics,Sitcom,22
1,1,2025-05-07,S02,The Detective,Crime Drama,50
2,1,2025-05-07,S08,Time Anomaly,Sci-Fi,48
3,2,2025-06-11,S04,Laugh Track,Sitcom,25
4,2,2025-06-11,S05,Office Antics,Sitcom,22


## 1. Exploratory Data Analysis (EDA)

The first step is to inspect the structure, dimensions, data types, missing values, and basic characteristics of the streaming-series dataset.

In [5]:
# Dataset dimensions
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types and non-null counts
print("\nDataset Information:")
df.info()

Number of rows: 8035
Number of columns: 6

Columns:
['ls dataviewing_session_id', 'transaction_date', 'show_id', 'item_name', 'category', 'avg_episode_minutes']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 8035 entries, 0 to 8034
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   ls dataviewing_session_id  8035 non-null   int64
 1   transaction_date           8035 non-null   str  
 2   show_id                    8035 non-null   str  
 3   item_name                  8035 non-null   str  
 4   category                   8035 non-null   str  
 5   avg_episode_minutes        8035 non-null   int64
dtypes: int64(2), str(4)
memory usage: 652.5 KB


In [6]:
# Missing values
print("Missing values per column:")
display(df.isnull().sum())

# Duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())

Missing values per column:


ls dataviewing_session_id    0
transaction_date             0
show_id                      0
item_name                    0
category                     0
avg_episode_minutes          0
dtype: int64


Number of duplicate rows: 0


In [7]:
print(df.columns.tolist())

['ls dataviewing_session_id', 'transaction_date', 'show_id', 'item_name', 'category', 'avg_episode_minutes']


In [8]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

['ls dataviewing_session_id', 'transaction_date', 'show_id', 'item_name', 'category', 'avg_episode_minutes']


In [9]:
df = df.rename(columns={
    "ls dataviewing_session_id": "viewing_session_id"
})

print(df.columns.tolist())

['viewing_session_id', 'transaction_date', 'show_id', 'item_name', 'category', 'avg_episode_minutes']


In [10]:
df.head()

,viewing_session_id,transaction_date,show_id,item_name,category,avg_episode_minutes
0,1,2025-05-07,S05,Office Antics,Sitcom,22
1,1,2025-05-07,S02,The Detective,Crime Drama,50
2,1,2025-05-07,S08,Time Anomaly,Sci-Fi,48
3,2,2025-06-11,S04,Laugh Track,Sitcom,25
4,2,2025-06-11,S05,Office Antics,Sitcom,22


In [11]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

print("Total records:", len(df))
print("Unique viewing sessions:", df["viewing_session_id"].nunique())
print("Unique shows:", df["show_id"].nunique())
print("Unique item names:", df["item_name"].nunique())
print("Unique transaction dates:", df["transaction_date"].nunique())

print("\nEarliest transaction date:", df["transaction_date"].min())
print("Latest transaction date:", df["transaction_date"].max())

Total records: 8035
Unique viewing sessions: 3500
Unique shows: 15
Unique item names: 15
Unique transaction dates: 181

Earliest transaction date: 2025-01-01 00:00:00
Latest transaction date: 2025-06-30 00:00:00


In [12]:
basket_sizes = df.groupby("viewing_session_id")["show_id"].nunique()

basket_sizes.describe()

count    3500.000000
mean        2.295714
std         0.847459
min         1.000000
25%         2.000000
50%         2.000000
75%         3.000000
max         4.000000
Name: show_id, dtype: float64

### Basket Size Distribution

A basket represents the unique shows watched during one viewing session. 
The basket-size distribution shows how many sessions contain one, two, three, or more unique shows.

In [13]:
basket_size_distribution = (
    basket_sizes
    .value_counts()
    .sort_index()
    .rename_axis("basket_size")
    .reset_index(name="number_of_sessions")
)

basket_size_distribution

,basket_size,number_of_sessions
0,1,513
1,2,1818
2,3,790
3,4,379


In [ ]:
!pip install --upgrade --force-reinstall matplotlib
import matplotlib.pyplot as plt

basket_size_distribution.plot(
    x="basket_size",
    y="number_of_sessions",
    kind="bar",
    legend=False
)

plt.title("Basket Size Distribution")
plt.xlabel("Number of Unique Shows per Session")
plt.ylabel("Number of Sessions")
plt.xticks(rotation=0)
plt.show()

## 2. Show Lookup Table

A separate lookup table is created to retain descriptive information about each show. 
This allows the transaction baskets to contain only show identifiers while show names, 
categories, and average episode runtimes remain available for interpretation.

In [ ]:
show_lookup = (
    df[
        ["show_id", "item_name", "category", "avg_episode_minutes"]
    ]
    .drop_duplicates()
    .sort_values("show_id")
    .reset_index(drop=True)
)

show_lookup

In [ ]:
print("Number of unique shows:", df["show_id"].nunique())
print("Rows in lookup table:", len(show_lookup))
print("Duplicate show IDs:", show_lookup["show_id"].duplicated().sum())

## 3. Building Transactions

For Association Rule Mining, the dataset is transformed from individual viewing records 
into transaction baskets. Each `viewing_session_id` represents one transaction, and the 
shows watched during that session form the items in the basket.

`transaction_date` is not included in the baskets because the association analysis focuses 
on combinations of shows watched within the same session.

In [ ]:
transactions = (
    df.groupby("viewing_session_id")["show_id"]
      .apply(lambda x: sorted(x.unique()))
      .reset_index(name="basket")
)

transactions.head(10)

In [ ]:
["viewing_session_id", "show_id"]

In [ ]:
print("Original viewing records:", len(df))
print("Number of transaction baskets:", len(transactions))
print("Unique viewing sessions:", df["viewing_session_id"].nunique())

transactions.head()

## 4. Transaction Validation

The prepared transactions are validated to ensure that each viewing session corresponds 
to exactly one basket and that no transaction dates or descriptive attributes are included 
in the baskets.

In [ ]:
print("Unique viewing sessions:", df["viewing_session_id"].nunique())
print("Transaction baskets:", len(transactions))
print("Unique shows:", df["show_id"].nunique())
print("Lookup table rows:", len(show_lookup))

assert len(transactions) == df["viewing_session_id"].nunique()
assert len(show_lookup) == df["show_id"].nunique()

print("\nValidation successful.")

In [ ]:
show_lookup.to_csv("../outputs/show_lookup.csv", index=False)

In [ ]:
transactions_export = transactions.copy()

transactions_export["basket"] = transactions_export["basket"].apply(
    lambda items: ",".join(items)
)

transactions_export.to_csv(
    "../outputs/session_transactions.csv",
    index=False
)

print("Files saved successfully.")

In [ ]:
import os

print(os.listdir("../outputs"))

## Role 1 Summary

The streaming-series dataset was successfully loaded and prepared for Association Rule Mining. 
Initial EDA was conducted to examine the dataset dimensions, viewing sessions, unique shows, 
transaction dates, missing values, duplicates, and basket-size distribution.

The data was then transformed into transaction baskets by grouping records according to 
`viewing_session_id`, with each session representing one basket of unique `show_id` values. 
`transaction_date` was excluded from the transaction baskets because it is not required for 
the association analysis.

A separate show lookup table was also created containing `show_id`, `item_name`, `category`, 
and `avg_episode_minutes`. The prepared transaction and lookup datasets were saved for use in 
the subsequent Association Rule Mining stages.

## Role 2: Choosing `min_support` and Generating Frequent Itemsets

In this section we build on the transaction baskets prepared by Member 1. The baskets are converted into a one-hot encoded format so that they can be used for association rule mining.

We then select a suitable `min_support` value based on the actual number of viewing sessions in our dataset. This helps ensure that the threshold is appropriate for our data rather than being chosen randomly.

Finally, we use the selected threshold to generate frequent itemsets and perform a basic check of the results. The resulting itemsets will then be passed to Member 3 for association rule generation and redundancy filtering.

In [29]:
!pip install mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori



## 2. Preparing baskets for itemset mining

Member 1's output (`session_transactions.csv`) stores each session's basket as a
comma-separated string, since lists can't be written directly to CSV. The first
step here is loading that file back in and restoring the basket column to an
actual Python list, so it can be one-hot encoded.

In [30]:
# --- Step 1: Load Member 1's transaction output ---
transactions = pd.read_csv("../outputs/session_transactions.csv")

# basket was flattened to a comma-separated string for CSV export — restore it to a list
transactions["basket"] = transactions["basket"].str.split(",")

print("Number of transaction baskets:", len(transactions))
transactions.head()

Number of transaction baskets: 3500


,viewing_session_id,basket
0,1,"[S02, S05, S08]"
1,2,"[S04, S05]"
2,3,"[S01, S02, S03]"
3,4,"[S09, S14]"
4,5,"[S07, S12]"


In [27]:
# --- Step 2: One-hot encode the baskets ---
# apriori() requires a wide True/False matrix, one column per item — not the
# list-of-lists format the baskets are currently in.
basket_list = transactions["basket"].tolist()

te = TransactionEncoder()
te_array = te.fit(basket_list).transform(basket_list)

onehot = pd.DataFrame(te_array, columns=te.columns_)

print("One-hot matrix shape:", onehot.shape)
onehot.head()

One-hot matrix shape: (3500, 15)


,S01,S02,S03,S04,S05,S06,S07,S08,S09,S10,S11,S12,S13,S14,S15
0,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False
1,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False
2,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
4,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False


## 3. Individual item support

Before choosing `min_support`, it helps to see how often each show is actually
watched across the 3500 sessions. This grounds the threshold decision in the
real distribution for this dataset, rather than guessing a round number.

In [31]:
# --- Step 3: Individual item support, translated into session counts ---
n_sessions = len(transactions)

item_support = onehot.mean().sort_values(ascending=False)

item_support_df = item_support.reset_index()
item_support_df.columns = ["show_id", "support"]
item_support_df["num_sessions"] = (item_support_df["support"] * n_sessions).round().astype(int)

item_support_df

,show_id,support,num_sessions
0,S01,0.225429,789
1,S09,0.222857,780
2,S06,0.158571,555
3,S14,0.157429,551
4,S11,0.157143,550
5,S05,0.156857,549
6,S10,0.156286,547
7,S04,0.155714,545
8,S07,0.154857,542
9,S02,0.153429,537


In [32]:
# --- Step 4: Trial a few candidate min_support values before committing ---
# With only 15 items, a threshold that's too low will make apriori's itemset
# count explode (too many combinations survive); too high and almost nothing
# will qualify. Trying several values first is how the "chosen from the
# observed distribution" part of Criterion 3 gets satisfied.
candidate_thresholds = [0.01, 0.02, 0.03, 0.05]

for min_sup in candidate_thresholds:
    itemsets = apriori(onehot, min_support=min_sup, use_colnames=True)
    required_sessions = round(min_sup * n_sessions)
    print(
        f"min_support={min_sup:<5} "
        f"(>= {required_sessions} of {n_sessions} sessions) "
        f"-> {len(itemsets)} itemsets"
    )


min_support=0.01  (>= 35 of 3500 sessions) -> 94 itemsets
min_support=0.02  (>= 70 of 3500 sessions) -> 31 itemsets
min_support=0.03  (>= 105 of 3500 sessions) -> 30 itemsets
min_support=0.05  (>= 175 of 3500 sessions) -> 22 itemsets


## 4. Final min_support decision

### Choosing the Minimum Support Threshold

Several candidate `min_support` values were tested against the 3,500 viewing sessions:

- `0.01` → 94 frequent itemsets (at least 35 sessions)
- `0.02` → 31 frequent itemsets (at least 70 sessions)
- `0.03` → 30 frequent itemsets (at least 105 sessions)
- `0.05` → 22 frequent itemsets (at least 175 sessions)

The threshold of `0.01` produced a much larger number of itemsets, while increasing the threshold beyond `0.02` resulted in relatively small reductions. Therefore `0.02` was selected as a reasonable balance between retaining recurring co-viewing patterns and removing very low-support patterns. This means an itemset must occur in at least 70 of the 3,500 viewing sessions to be considered frequent.

In [33]:
# --- Step 5: Generate final frequent itemsets at the chosen threshold ---
FINAL_MIN_SUPPORT = 0.02   # <- replace with whichever value you justified above

frequent_itemsets = apriori(onehot, min_support=FINAL_MIN_SUPPORT, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)


In [34]:
# --- Step 6: Sanity check on itemset sizes ---
# Confirms the mined itemsets aren't all singletons (which would mean the
# threshold is too high to find any real co-viewing pairs/triples).
frequent_itemsets["itemset_size"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets["itemset_size"].value_counts().sort_index()

itemset_size
1    15
2    13
3     3
Name: count, dtype: int64

In [35]:
# --- Step 7: Save output for Member 3 (thresholds & redundancy removal) ---
frequent_itemsets.to_pickle("../outputs/frequent_itemsets.pkl")

print("Frequent itemsets saved.")

Frequent itemsets saved.


## Role 2 Summary 

In this section Member 1's session transaction output was loaded and the viewing baskets were restored into lists of shows. The baskets were then converted into a one-hot encoded transaction matrix using `TransactionEncoder` where each row represents a viewing session and each column represents a show.

Several candidate `min_support` values were tested using the 3,500 viewing sessions. The results were 94 frequent itemsets at 0.01, 31 at 0.02, 30 at 0.03, and 22 at 0.05. Based on the observed distribution, `min_support = 0.02` was selected as a reasonable threshold. This requires an itemset to appear in at least 70 of the 3,500 sessions to be considered frequent.

Finally, the frequent itemsets were generated using the Apriori algorithm, sorted by support, checked by itemset size, and saved as `frequent_itemsets.pkl` for use in the next stage of the analysis.
